# StreamVault Licensing Analysis — Submission Notebook

This notebook presents the project deliverables: acquisition accounting, a data-quality audit, cleaning rules, NumPy array operations, stakeholder cohort comparisons, and the licensing decision workflow. It uses saved project data and makes no API calls or credential reads.

TMDb ratings and popularity are audience signals only. They are not licensing prices, rights availability, or financial return.

## Deliverable 1 — Load data and account for source records

The notebook expects to run from the project root or its `notebooks` folder. The acquisition log is used to account for every source record.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "processed" / "tmdb_movies.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "data" / "processed" / "tmdb_movies.csv").exists():
    raise FileNotFoundError("Run this notebook from the project root or notebooks folder")

sys.path.insert(0, str(PROJECT_ROOT))
from src.analyze import build_content_shortlist, build_data_quality_summary, build_rating_summary

movies = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "tmdb_movies.csv")
print("Movie table shape:", movies.shape)
print("Columns:", ", ".join(movies.columns))
print(movies[["tmdb_id", "title", "release_year", "vote_average", "vote_count"]].head().to_string(index=False))

Movie table shape: (112, 22)
Columns: tmdb_id, title, original_title, release_date, runtime, budget, revenue, status, genres, production_companies, spoken_languages, popularity, vote_average, vote_count, adult, top_cast, director, release_year, collection_id, collection_name, collection_metadata_status, top_billed_actor
 tmdb_id                      title  release_year  vote_average  vote_count
     155            The Dark Knight          2008         8.500       37437
     165 Back to the Future Part II          1989         7.784       14540
     242     The Godfather Part III          1990         7.413        7205
   49026      The Dark Knight Rises          2012         7.801       25350
    8077                     Alien³          1992         6.391        6730


In [2]:
source_path = PROJECT_ROOT / "data" / "raw" / "id_list.csv"
log_path = PROJECT_ROOT / "data" / "processed" / "tmdb_acquisition_log.csv"
source_ids = pd.read_csv(source_path, dtype={"tmdb_id": "string"}, keep_default_na=False)

if log_path.exists():
    acquisition_log = pd.read_csv(log_path)
    print("Acquisition outcomes:")
    print(acquisition_log["status"].value_counts(dropna=False).to_string())
    assert len(source_ids) == len(acquisition_log)
    assert acquisition_log["source_record"].is_unique
    assert not acquisition_log["status"].eq("pending").any()
else:
    print("No acquisition log yet. Run: python -m src.acquire")

print("Source records:", len(source_ids))
print("Acquired movies:", len(movies))

Acquisition outcomes:
status
success      112
duplicate      6
api_error      5
invalid        3
Source records: 126
Acquired movies: 112


## Deliverable 2 — NumPy arrays: list versus array, shape, and dtype

A Python list can hold mixed object types and usually needs element-by-element work. A NumPy array has a regular shape and dtype, which enables compact vectorized numeric operations. `shape` reports the size of each dimension; `size` is the total number of elements.

In [3]:
rating_list = movies["vote_average"].tolist()
rating_array = np.asarray(rating_list, dtype=np.float64)
vote_count_array = pd.to_numeric(movies["vote_count"], errors="coerce").to_numpy(dtype=np.float64)
popularity_array = pd.to_numeric(movies["popularity"], errors="coerce").to_numpy(dtype=np.float64)
feature_array = np.column_stack((rating_array, vote_count_array, popularity_array))

print("List type and length:", type(rating_list).__name__, len(rating_list))
print("Rating array: ndim =", rating_array.ndim, "shape =", rating_array.shape, "dtype =", rating_array.dtype)
print("Feature array: ndim =", feature_array.ndim, "shape =", feature_array.shape, "size =", feature_array.size)
print("First three ratings doubled with a list:", [value * 2 for value in rating_list[:3]])
print("First three ratings doubled with NumPy:", (rating_array[:3] * 2).tolist())

List type and length: list 112
Rating array: ndim = 1 shape = (112,) dtype = float64
Feature array: ndim = 2 shape = (112, 3) size = 336
First three ratings doubled with a list: [17.0, 15.568, 14.826]
First three ratings doubled with NumPy: [17.0, 15.568, 14.826]


## Deliverable 3 — Data-quality checks and cleaning decisions

The cleaner rejects malformed IDs, trims titles, and invalidates impossible numeric values. Zero budgets and revenues are preserved rather than assumed to mean missing; the report counts them for review. Cleaning should not silently turn an unknown value into a fact.

In [4]:
quality = build_data_quality_summary(movies)
print(quality.to_string(index=False))

numeric_checks = {}
for column in ("budget", "revenue", "runtime", "vote_average", "vote_count"):
    values = pd.to_numeric(movies[column], errors="coerce")
    numeric_checks[column] = {
        "missing": int(values.isna().sum()),
        "zero": int(values.eq(0).sum()),
        "negative": int(values.lt(0).sum()),
    }
print(pd.DataFrame(numeric_checks).T.to_string())

                                   metric  value
                                film_rows    112
                       duplicate_tmdb_ids      0
                     missing_release_year      0
                        films_before_2000     41
                        films_2000_onward     71
                    earliest_release_year   1966
                      latest_release_year   2019
                       zero_budget_values      0
                      zero_revenue_values      0
                   missing_values_tmdb_id      0
                     missing_values_title      0
            missing_values_original_title      0
              missing_values_release_date      0
                   missing_values_runtime      0
                    missing_values_budget      0
                   missing_values_revenue      0
                    missing_values_status      0
                    missing_values_genres      0
      missing_values_production_companies      0
          missing_va

## Deliverable 4 — Compare Finance cohorts

The all-years view and the 2000-onward view are reported separately. Vote-weighted averages give more influence to titles with more votes; they describe TMDb audience ratings, not expected licensing performance.

In [5]:
rating_summary = build_rating_summary(movies)
print(rating_summary.to_string(index=False))

     cohort  film_count  mean_vote_average  median_vote_average  vote_weighted_average  total_votes
  all_years         112            7.58792                7.720               7.807562      1965880
2000_onward          71            7.43307                7.491               7.682293      1293062


## Deliverable 5 — Audience shortlist, talent signals, and licensing scenario

The adjusted rating shrinks low-vote ratings toward the dataset mean. This cell regenerates the recommendations from saved data, then evaluates the current offer template. It makes no API calls; licensing remains `no_eligible_offers` until verified price and rights terms are entered.

In [6]:
shortlist = build_content_shortlist(movies)
print("Audience shortlist:")
print(shortlist[["cohort", "rank", "tmdb_id", "title", "release_year", "vote_average", "vote_count", "adjusted_rating"]].to_string(index=False))

from src.recommend import run_recommendations
from src.licensing import run_license_decision

recommendations = run_recommendations(
    movies_path=PROJECT_ROOT / "data" / "processed" / "tmdb_movies.csv",
    acquisition_log_path=PROJECT_ROOT / "data" / "processed" / "tmdb_acquisition_log.csv",
    offers_path=PROJECT_ROOT / "data" / "raw" / "license_offers_template.csv",
)
print("\nFranchise targets:")
print(recommendations["franchises"].to_string(index=False))
print("\nStandalone shortlist:")
print(recommendations["standalones"].to_string(index=False))
print("\nRepeated talent signals (top three per cohort and role):")
talent_summary = recommendations["talent"].groupby(["cohort", "role"], sort=False).head(3)
print(talent_summary.to_string(index=False))

license_summary, license_candidates = run_license_decision(
    offers_path=PROJECT_ROOT / "data" / "raw" / "license_offers_template.csv"
)
print("\nBudget-constrained offer scenario:")
print(license_summary.to_string(index=False))

Audience shortlist:
     cohort  rank  tmdb_id                                             title  release_year  vote_average  vote_count  adjusted_rating
  all_years     1      278                          The Shawshank Redemption          1994         8.735       31890         8.354065
  all_years     2      238                                     The Godfather          1972         8.689       24020         8.251174
  all_years     3   157336                                      Interstellar          2014         8.493       41967         8.244812
  all_years     4      155                                   The Dark Knight          2008         8.500       37437         8.228633
  all_years     5      680                                      Pulp Fiction          1994         8.480       31373         8.180505
  all_years     6      122     The Lord of the Rings: The Return of the King          2003         8.508       27867         8.174336
  all_years     7       13                

## Deliverable 6 — Ranked diagnosis and complete candidate reconciliation

The diagnosis ranks risks by impact. The acquisition log accounts for every source record, while cached records and unresolved IDs remain distinguishable.

In [7]:
print(recommendations["diagnosis"].to_string(index=False))

acquisition_log = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "tmdb_acquisition_log.csv")
print("\nAcquisition status counts:")
print(acquisition_log["status"].value_counts(dropna=False).to_string())
unresolved_rows = acquisition_log.loc[
    acquisition_log["status"].eq("api_error")
    & ~acquisition_log["used_cached_data"].fillna(False).astype(bool),
    ["source_record", "tmdb_id", "reason"],
]
print("\nValid IDs still unresolved (impact cannot be estimated from available data):")
print(unresolved_rows.to_string(index=False))

 risk_rank severity                                   issue                                                                                                                       evidence  candidates_affected                                                                                                                        recommendation_impact                                                            action
         1 Critical            No verified licensing offers                                0 of 13 negotiation candidates have all required price, rights, scope, date, and source fields.                   13                        No $14M bid or return can be validated; the current portfolio model must not be treated as a purchase recommendation. Obtain itemized fees and confirmed rights terms for each package.
         2     High       Valid candidate IDs not retrieved                                                      112 of 117 unique valid IDs are in the current data; 5 re

## Deliverable 7 — Client memo, decision notes, and improvement

The memo is kept concise for a one-page brief. The decision notes preserve assumptions, the 500-title re-run boundary, and the quarterly cached-data improvement.

In [8]:
memo_path = PROJECT_ROOT / "reports" / "client_memo.md"
decision_notes_path = PROJECT_ROOT / "reports" / "decision_notes.md"
print("CLIENT MEMO\n")
print(memo_path.read_text(encoding="utf-8"))
print("\nDECISION NOTES\n")
print(decision_notes_path.read_text(encoding="utf-8"))

CLIENT MEMO

# Memo to Kwame

**From:** Content Analysis Team  
**Date:** 20 November 2026  
**Subject:** Negotiation priorities for the $14M content budget

## What I found

Scores account for vote volume. All-years averages: **8.22**
for the top five standalones versus **7.99** for the top
three franchises. Since 2000, the leaders are **The Lord of the Rings Collection** and
**Interstellar**; across all years, they are The Lord of the Rings Collection
and The Shawshank Redemption.

## What I built

| What you asked for | Status | Evidence |
|---|---|---|
| Compare both time periods | Complete | 112 films; all years and post-1999. |
| Account for candidate records | Gaps noted | 126 rows: 112 fetched, 0 reused, 6 duplicates, 3 invalid, 5 unresolved. |
| Shortlists and talent signals | Complete | Three franchises, five standalones, recurring directors and actors. |
| Check the budget | Waiting for offers | 0/13 offers have complete terms. |

## What I recommend

Quote standalones first

## Assumptions recorded for submission

The budget scenario maximizes summed adjusted audience scores under the $14M cap; it is not an ROI objective. It assumes verified offers are comparable for the intended territory and window. Franchise membership requires confirmed collection metadata, and the top-billed actor is a billing-order proxy. The offline 500-title test checks scaling; live API failures remain subject to reconciliation. A cached-data fallback prevents transient refresh failures from silently removing previously acquired titles.